# Simple RAG 실습 - 자동차 카탈로그
자동차 카탈로그를 **조각으로 나누고(분할) → 벡터 DB에 저장하고(임베딩) → 질문과 가까운 조각을 찾아(검색) → LLM에 넘겨 답하게(생성)** 합니다.

같은 내용을 txt와 PDF 두 형식으로 준비했습니다. `SOURCE` 값을 바꿔 두 형식을 모두 실습할 수 있습니다.
- `car_catalog.txt`: 줄바꿈이 그대로 있어 바로 사용
- `car_catalog.pdf`: 추출하면 줄바꿈이 사라지고 공백이 탭으로 나오므로 정제가 필요

In [ ]:
# 최초 실행 시 필수 패키지 설치
# faiss-cpu: 벡터 검색 / langchain-*: RAG 구성 도구 / langchain-classic: RetrievalQA 등 체인
# pypdf: PDF 읽기 / python-dotenv: .env 읽기
!pip install -U faiss-cpu langchain langchain-classic langchain-core langchain-community langchain-openai langchain-text-splitters pypdf python-dotenv

In [3]:
import os                       # 환경 변수 읽기
import re                       # 정규식으로 PDF 텍스트 정제
from pathlib import Path        # 파일 경로 다루기
from dotenv import load_dotenv  # .env 파일 읽기

from langchain_core.documents import Document                              # 텍스트 + 메타데이터를 담는 문서 객체
from langchain_community.document_loaders import PyPDFLoader, TextLoader   # PDF, txt 파일 읽기
from langchain_text_splitters import RecursiveCharacterTextSplitter        # 문서를 조각으로 나누기
from langchain_community.vectorstores import FAISS                         # 벡터 DB
from langchain_openai import OpenAIEmbeddings, ChatOpenAI                  # 임베딩 모델, 대화 모델
from langchain_core.prompts import ChatPromptTemplate                      # 프롬프트 템플릿

# LangChain 1.x에서 체인(RetrievalQA 등)은 langchain_classic 패키지로 이동했습니다.
try:
    from langchain_classic.chains import RetrievalQA, create_retrieval_chain
    from langchain_classic.chains.combine_documents import create_stuff_documents_chain
except ImportError:
    # langchain_classic이 없는 이전 버전(0.x) 환경에서는 기존 경로 사용
    from langchain.chains import RetrievalQA, create_retrieval_chain
    from langchain.chains.combine_documents import create_stuff_documents_chain

# 현재 폴더의 .env를 먼저 찾고, 없으면 상위 폴더의 .env 사용
load_dotenv(Path.cwd() / '.env') or load_dotenv(Path.cwd().parent / '.env')
openai_api_key = os.getenv("OPENAI_API_KEY")
# 키가 없거나 .env.example의 예시 문구를 그대로 둔 경우 실행을 멈춤
if not openai_api_key or openai_api_key.startswith('운영팀에서_'):
    raise ValueError("OPENAI_API_KEY가 설정되지 않았습니다. .env 파일 또는 환경 변수를 확인해주세요.")

MODEL = "gpt-4o-mini"                     # 답변을 생성할 대화 모델
EMBED_MODEL = "text-embedding-3-small"    # 문장을 숫자 벡터로 바꾸는 임베딩 모델

SOURCE = "txt"   # "txt" 또는 "pdf"

## 1. 문서 불러오기
PDF에서 추출한 텍스트는 다음과 같은 형태입니다.
```text
[Hyundai\tAvante]현대자동차의\t대표적인\t준중형\t세단입니다.\t\t1.6리터\t...특징입니다.[Kia\tSorento]...
```
- 띄어쓰기는 탭 1개(`\t`), 원래 줄바꿈은 탭 2개(`\t\t`)로 나옵니다.
- 차량 섹션(`[모델명]`) 사이에 빈 줄이 없어 분할기가 차량 단위로 나눌 수 없습니다.
- 페이지 경계에서 문장이 끊겨 있습니다.

`clean_pdf_catalog_text`는 이 규칙을 되돌려 txt와 같은 형태로 만듭니다. pypdf 버전에 따라 줄바꿈(`\n`)이 섞여 나오는 경우도 함께 처리합니다.

In [4]:
def clean_pdf_catalog_text(raw: str) -> str:
    """PDF 추출 텍스트를 txt 원본과 같은 형태로 복원"""
    text = re.sub(r"\t\t\n?", "\x00", raw)   # 탭 2개 = 원래 줄바꿈 (임시 표시)
    text = text.replace("\t\n", " ")         # 어절 경계에서 줄이 넘어간 경우
    text = text.replace("\n", "")             # 나머지 줄바꿈은 PDF 자동 줄넘김
    text = text.replace("\t", " ")            # 탭 1개 = 띄어쓰기
    text = text.replace("\x00", "\n")         # 임시 표시를 실제 줄바꿈으로 되돌림
    text = text.replace("3초대의제로백", "3초대의 제로백")   # PDF에서 붙어 버린 단어 보정

    # 차량 섹션 제목을 한 줄로 떼어 내고, 섹션 사이에 빈 줄 추가
    # 예: "...특징입니다.[Kia Sorento]기아자동차의..." → "...특징입니다.\n\n[Kia Sorento]\n기아자동차의..."
    text = re.sub(r"\s*(\[[^\]]+\])\s*", r"\n\n\1\n", text)
    text = re.sub(r" {2,}", " ", text)        # 연속된 공백을 하나로
    text = re.sub(r" *\n *", "\n", text)      # 줄 앞뒤에 남은 공백 제거
    return text.strip()


def load_catalog_documents(source: str = "txt") -> list[Document]:
    """car_catalog.txt 또는 car_catalog.pdf를 읽어 Document 리스트로 반환"""
    if source == "txt":
        path = Path("car_catalog.txt")
        if not path.exists():
            raise FileNotFoundError(f"{path} 파일을 찾을 수 없습니다.")
        # txt는 줄바꿈이 그대로 있으므로 정제 없이 바로 사용
        return TextLoader(str(path), encoding="utf-8").load()

    if source == "pdf":
        path = Path("car_catalog.pdf")
        if not path.exists():
            raise FileNotFoundError(f"{path} 파일을 찾을 수 없습니다.")
        # PDF는 페이지마다 Document 하나씩 반환됨
        pages = PyPDFLoader(str(path)).load()
        # 페이지 경계에서 끊긴 문장을 잇기 위해 전체를 합친 뒤 정제
        merged = clean_pdf_catalog_text(" ".join(p.page_content for p in pages))
        # 정제한 전체 텍스트를 Document 하나로 만들어 반환
        return [Document(page_content=merged, metadata={"source": str(path)})]

    raise ValueError('SOURCE는 "txt" 또는 "pdf"로 지정하세요.')


# 위에서 정한 SOURCE 형식으로 문서를 읽고 앞부분 300자 확인
docs = load_catalog_documents(SOURCE)
print(docs[0].page_content[:300])

[Hyundai Avante]
현대자동차의 대표적인 준중형 세단입니다.
1.6리터 가솔린 엔진, IVT 무단변속기를 탑재하였으며
연비는 약 15.4km/L입니다.
스마트 센스, 전방 충돌방지 보조, 차로 유지 보조 등의 첨단 안전 기능이 포함되어 있습니다.
젊은 층과 패밀리 고객을 위한 경제적이면서도 세련된 디자인이 특징입니다.

[Kia Sorento]
기아자동차의 중형 SUV로, 디젤 및 하이브리드 모델이 있습니다.
최대 7인승까지 탑승 가능하며 넓은 실내 공간과 높은 적재력을 자랑합니다.
12.3인치 파노라믹 디스플레이와 다양


In [5]:
# 검증: PDF를 정제한 결과가 txt 원본과 같아야 함 (API 호출 없음)
# 두 형식을 모두 읽어 비교
txt_text = load_catalog_documents("txt")[0].page_content.strip()
pdf_text = load_catalog_documents("pdf")[0].page_content.strip()

# assert: 조건이 거짓이면 뒤의 메시지와 함께 오류를 내고 멈춤
assert pdf_text.count("[") == 5, "차량 섹션 5개가 모두 있어야 함"
assert "\t" not in pdf_text, "탭이 남아 있으면 안 됨"
assert pdf_text == txt_text, "PDF 정제 결과가 txt 원본과 다릅니다."
print("PDF 정제 결과 = txt 원본, 검증 통과")

PDF 정제 결과 = txt 원본, 검증 통과


## 2. 분할 → 임베딩 → 벡터 DB

In [6]:
# 문서를 검색하기 좋은 크기의 조각(chunk)으로 나누기
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,       # 조각 하나의 최대 글자 수
    chunk_overlap=50,     # 앞뒤 조각이 겹치는 글자 수 (문장이 잘려도 맥락 유지)
    separators=["\n\n", "\n", " ", ""],   # 빈 줄(차량 섹션) 기준으로 먼저 나눔
)
splits = splitter.split_documents(docs)

# 나뉜 조각 수와 각 조각의 첫 줄(차량 이름) 확인
print(f"{len(splits)}개 조각")
for d in splits:
    print(" -", d.page_content.split("\n")[0])

# 각 조각을 임베딩(숫자 벡터)으로 바꿔 FAISS 벡터 DB에 저장
embedding = OpenAIEmbeddings(model=EMBED_MODEL, api_key=openai_api_key)
vectordb = FAISS.from_documents(splits, embedding)

# 검색기: 질문과 가장 가까운 조각 3개(k=3)를 찾아 줌
retriever = vectordb.as_retriever(search_kwargs={"k": 3})
# 답변을 생성할 대화 모델 (temperature=0: 매번 같은 답변이 나오도록)
llm = ChatOpenAI(model=MODEL, temperature=0, api_key=openai_api_key)

5개 조각
 - [Hyundai Avante]
 - [Kia Sorento]
 - [Genesis G80]
 - [Hyundai Ioniq 5]
 - [Kia EV6]


## 3. 방식 A: RetrievalQA
검색과 답변 생성을 한 번에 처리하는 이전 방식의 체인입니다. 구조가 단순해 RAG 흐름을 처음 볼 때 이해하기 쉽습니다.

In [ ]:
query = "아이오닉 5에 대해 설명해줘"

# RetrievalQA: 검색(retriever) → 프롬프트 구성 → 답변 생성(llm)을 한 번에 처리
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",              # 검색된 조각을 모두 프롬프트에 넣는 방식
    retriever=retriever,
    return_source_documents=True,    # 답변에 사용한 조각도 함께 반환
)

response = qa_chain.invoke({"query": query})
print("[RetrievalQA 응답]")
print(response["result"])            # 생성된 답변
# 답변 근거로 사용된 조각의 첫 줄(차량 이름)
print("\n참조 조각:", [d.page_content.split("\n")[0] for d in response["source_documents"]])

## 4. 방식 B: create_retrieval_chain
프롬프트를 직접 지정하는 방식입니다. 다음 수업(4번)에서는 체인 없이 검색 결과를 직접 프롬프트에 넣어 같은 일을 해 봅니다.

In [ ]:
# 프롬프트를 직접 지정 ({context}: 검색된 조각이 들어갈 자리, {input}: 질문이 들어갈 자리)
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "다음 자동차 카탈로그 내용(Context)만을 바탕으로 질문에 정확하고 자연스러운 한국어로 답변하세요.\n"
        "Context에 없는 내용이면 '카탈로그에서 찾을 수 없습니다.'라고 답하세요.\n\n[Context]\n{context}",
    ),
    ("human", "{input}"),
])
# 검색된 조각들을 프롬프트에 채워 넣어 답변을 만드는 체인
combine_docs_chain = create_stuff_documents_chain(llm, prompt)
# 검색 체인 + 답변 체인을 연결
rag_chain = create_retrieval_chain(retriever, combine_docs_chain)

# 카탈로그에 있는 질문 2개와 없는 질문 1개("쏘나타 가격")를 비교
for q in ["아이오닉 5에 대해 설명해줘", "가족 여행용으로 적합한 차는?", "쏘나타 가격은?"]:
    result = rag_chain.invoke({"input": q})
    print("Q:", q)
    print("A:", result["answer"], "\n")